<a href="https://colab.research.google.com/github/MuhammadAyyanHassan/flyrank-ml-internship-work/blob/main/w03_data_contract_ready.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAyyanHassan/flyrank-ml-internship-work/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane:** Refresh / Content Opportunity Scoring  
**Decision month:** March 2026  
**Outcome month:** April 2026

This notebook defines and verifies the data contract before feature/model work. The source fact table is daily; the decision frame aggregates the March window to one row per content-client pair and uses the following month only for the future outcome label.


## 0. Setup and warehouse access

We query the gated warehouse remotely with DuckDB rather than downloading the 79M-row fact table. The Hugging Face token is read from Colab Secrets as `HF_TOKEN`; it is never written into notebook source.


In [12]:
# Install/query dependencies.
%pip -q install duckdb scikit-learn

import os
import duckdb
import pandas as pd
import numpy as np

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is missing. Add a Read token in Colab Secrets with the name HF_TOKEN.")

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

MARCH_REL = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
APRIL_REL = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet"

print("Warehouse connection configured.")
print("Development window:", "2026-03")
print("Future outcome window:", "2026-04")


Warehouse connection configured.
Development window: 2026-03
Future outcome window: 2026-04


## 0.1 Schema check

Before writing feature SQL, inspect the real March partition schema. This is a guard against guessing column names.


In [13]:
schema = con.sql(
    f"DESCRIBE SELECT * FROM read_parquet('{MARCH_REL}')"
).df()

display(schema[["column_name", "column_type"]])


,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


## 0.2 Resolve the warehouse column names

The repository documentation specifies the key semantic fields, while this cell resolves the actual physical names from the partition schema. If a required field cannot be found, the notebook stops instead of silently substituting the wrong column.


In [14]:
available = set(schema["column_name"].tolist())

def resolve(candidates, label):
    for name in candidates:
        if name in available:
            return name
    raise KeyError(f"Could not resolve {label}. Available columns include: {sorted(available)}")

DATE_COL = resolve(["report_date"], "report date")
CLIENT_COL = resolve(["client_hash_id", "client_id"], "client key")
CONTENT_COL = resolve(["content_hash_id", "content_id"], "content key")
GSC_AVAIL_COL = resolve(["gsc_data_available"], "GSC availability flag")
IMP_COL = resolve(["gsc_impressions", "impressions"], "GSC impressions")
CLICK_COL = resolve(["gsc_clicks", "clicks"], "GSC clicks")
POSITION_COL = resolve(["gsc_avg_position", "avg_position"], "GSC average position")

resolved = pd.DataFrame({
    "semantic_field": [
        "report_date", "client key", "content key",
        "GSC availability", "GSC impressions", "GSC clicks", "GSC average position"
    ],
    "physical_column": [
        DATE_COL, CLIENT_COL, CONTENT_COL,
        GSC_AVAIL_COL, IMP_COL, CLICK_COL, POSITION_COL
    ]
})
display(resolved)


,semantic_field,physical_column
0,report_date,report_date
1,client key,client_hash_id
2,content key,content_hash_id
3,GSC availability,gsc_data_available
4,GSC impressions,gsc_impressions
5,GSC clicks,gsc_clicks
6,GSC average position,gsc_avg_position


# 1. Unit of analysis + time window

**Source grain:** one row represents one pseudonymized content item for one pseudonymized client on one `report_date` in `fact_content_daily_performance`.

**Decision grain:** for this lane, the March decision frame becomes one row per content-client pair after aggregating the March 1–31 daily observations.

**Prediction/ranking target:** rank content items by their risk/opportunity for a **future search-impression decline in April 2026**. The label is a future outcome proxy defined as April impressions falling by more than 20% versus March impressions, for content with positive March impressions.

**Feature window:** March 1–31, 2026.  
**Outcome window:** April 1–30, 2026.

This separation matters: March features are knowable at the March decision point; April performance is not.

The raw fact-table grain and the decision-frame grain are intentionally distinguished rather than conflated.


In [15]:
# Verification query 1 — raw grain: duplicate content-client-date rows.
grain_query = f"""
SELECT
    {CLIENT_COL} AS client_hash_id,
    {CONTENT_COL} AS content_hash_id,
    {DATE_COL} AS report_date,
    COUNT(*) AS row_count
FROM read_parquet('{MARCH_REL}')
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5
"""

grain_check = con.sql(grain_query).df()
display(grain_check)

assert grain_check.empty, "Grain check failed: duplicate content-client-date rows were found."
print("PASS: no duplicate content-client-date combinations were found in the March partition.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,row_count


PASS: no duplicate content-client-date combinations were found in the March partition.


# 2. Fields: feature / label / context / excluded

### Features — March decision window

The final feature frame will contain at most five aggregated March signals:

1. `march_impressions` — total GSC impressions in March.
2. `march_clicks` — total GSC clicks in March.
3. `march_ctr_pct` — clicks divided by impressions × 100.
4. `march_avg_position` — impression-weighted average GSC position.
5. `march_impression_days` — number of March days with at least one impression.

Each is observable by the March decision point.

### Label / proxy

`future_decline_label` = 1 when April impressions are more than 20% below March impressions, with a positive March baseline.

This is a future observed outcome proxy, not a claim about an eventual business decision.

### Context

`client_hash_id`, `content_hash_id`, and `report_date` are identifiers/time keys used for grouping, joining, and window definition. They are not predictive features.

### Excluded

- April performance fields from the feature set: they occur after the decision point.
- `trend_pct` / `trend_direction` or any equivalent label-derived field: they directly encode decline information.
- IDs as model features: pseudonymous identifiers have no meaningful numeric semantics.
- Product decision flags/scores, if encountered: they would reproduce an existing decision rather than discover independent signal.


In [16]:
# Verification query 2 — March slice size and date span.
count_window_query = f"""
SELECT
    COUNT(*) AS row_count,
    MIN({DATE_COL}) AS min_report_date,
    MAX({DATE_COL}) AS max_report_date
FROM read_parquet('{MARCH_REL}')
"""

count_window = con.sql(count_window_query).df()
display(count_window)

assert str(count_window.loc[0, "min_report_date"])[:10] == "2026-03-01", "Unexpected March minimum date."
assert str(count_window.loc[0, "max_report_date"])[:10] == "2026-03-31", "Unexpected March maximum date."
print("PASS: March partition spans exactly 2026-03-01 through 2026-03-31.")


,row_count,min_report_date,max_report_date
0,9841378,2026-03-01,2026-03-31


PASS: March partition spans exactly 2026-03-01 through 2026-03-31.


## 2.1 Missingness and availability

Availability is handled explicitly. The warehouse uses a boolean availability flag that can contain NULL, so `IS TRUE` is intentional.

The same query reports how many March rows survive the GSC-availability filter. This avoids treating an unavailable measurement as a genuine zero.


In [17]:
# Verification query 3 — availability using IS TRUE.
availability_query = f"""
SELECT
    COUNT(*) AS all_march_rows,
    COUNT(*) FILTER (WHERE {GSC_AVAIL_COL} IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE {GSC_AVAIL_COL} IS NOT TRUE) AS gsc_not_available_rows,
    COUNT(*) FILTER (WHERE {GSC_AVAIL_COL} IS NULL) AS gsc_null_flag_rows
FROM read_parquet('{MARCH_REL}')
"""

availability = con.sql(availability_query).df()
display(availability)

print(
    f"GSC-available March rows: {int(availability.loc[0, 'gsc_available_rows']):,} "
    f"of {int(availability.loc[0, 'all_march_rows']):,} total rows."
)
print("PASS: availability was checked with IS TRUE; NULL flags were not silently treated as available.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,all_march_rows,gsc_available_rows,gsc_not_available_rows,gsc_null_flag_rows
0,9841378,3611061,6230317,0


GSC-available March rows: 3,611,061 of 9,841,378 total rows.
PASS: availability was checked with IS TRUE; NULL flags were not silently treated as available.


# 3. Five-feature frame

The frame below is built only from March observations. Each row is one content-client pair.

The five features are deliberately simple and decision-time safe:

- **`march_impressions`** — knowable at the March decision point because these are accumulated GSC impressions through March 31.
- **`march_clicks`** — knowable at the decision point because they are accumulated GSC clicks through March 31.
- **`march_ctr_pct`** — knowable at the decision point because it is computed only from March clicks and March impressions.
- **`march_avg_position`** — knowable at the decision point because it is computed from March GSC position observations only.
- **`march_impression_days`** — knowable at the decision point because it counts March days with observed impressions.

The future April outcome is kept separate from these features.


In [18]:
feature_sql = f"""
WITH march AS (
    SELECT
        {CLIENT_COL} AS client_hash_id,
        {CONTENT_COL} AS content_hash_id,
        SUM({IMP_COL}) AS march_impressions,
        SUM({CLICK_COL}) AS march_clicks,
        CASE
            WHEN SUM({IMP_COL}) > 0
            THEN 100.0 * SUM({CLICK_COL}) / SUM({IMP_COL})
            ELSE NULL
        END AS march_ctr_pct,
        SUM(
            CASE
                WHEN {IMP_COL} > 0 AND {POSITION_COL} > 0 THEN {IMP_COL} * {POSITION_COL}
                ELSE 0
            END
        ) / NULLIF(
            SUM(
                CASE
                    WHEN {IMP_COL} > 0 AND {POSITION_COL} > 0 THEN {IMP_COL}
                    ELSE 0
                END
            ), 0
        ) AS march_avg_position,
        COUNT(DISTINCT CASE WHEN {IMP_COL} > 0 THEN {DATE_COL} END) AS march_impression_days
    FROM read_parquet('{MARCH_REL}')
    WHERE {GSC_AVAIL_COL} IS TRUE
    GROUP BY 1, 2
),
april AS (
    SELECT
        {CLIENT_COL} AS client_hash_id,
        {CONTENT_COL} AS content_hash_id,
        SUM({IMP_COL}) AS april_impressions
    FROM read_parquet('{APRIL_REL}')
    WHERE {GSC_AVAIL_COL} IS TRUE
    GROUP BY 1, 2
)
SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.march_impressions,
    m.march_clicks,
    m.march_ctr_pct,
    m.march_avg_position,
    m.march_impression_days,
    a.april_impressions,
    CASE
        WHEN m.march_impressions > 0
             AND a.april_impressions < 0.80 * m.march_impressions
        THEN 1
        ELSE 0
    END AS future_decline_label
FROM march m
INNER JOIN april a
    USING (client_hash_id, content_hash_id)
"""

feature_frame = con.sql(feature_sql).df()
feature_frame["april_impressions"] = feature_frame["april_impressions"].fillna(0)

print(f"Decision rows: {len(feature_frame):,}")
print(f"Future decline rate: {feature_frame['future_decline_label'].mean():.3f}")
display(feature_frame.head(10))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Decision rows: 158,549
Future decline rate: 0.478


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_ctr_pct,march_avg_position,march_impression_days,april_impressions,future_decline_label
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,0.107313,6.893301,31,6787.0,0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,0.000000,3.433962,31,405.0,0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,0.106572,6.535346,31,8475.0,0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,0.262945,7.435680,31,6091.0,0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,0.233100,3.983213,31,287.0,1
5,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,223.0,1.0,0.448430,10.681818,31,552.0,0
6,client_73cda7b4e4f265ea,content_1f380a642aed423b,96.0,1.0,1.041667,7.311688,31,111.0,0
7,client_73cda7b4e4f265ea,content_22c063002b7c1caf,314.0,1.0,0.318471,10.057325,31,322.0,0
8,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,7709.0,20.0,0.259437,5.127643,31,5261.0,1
9,client_73cda7b4e4f265ea,content_20403327d8d9374c,3561.0,10.0,0.280820,9.273799,31,4197.0,0


## 3.1 Leakage trap — deliberately add one label-derived feature

Now we intentionally do the wrong thing.

`leaked_decline_score` is copied directly from the future label. It is **not** a legitimate feature. This is a controlled demonstration that a model can appear nearly perfect when future/label information is accidentally exposed.

We use a quick train/test score only to show the trap; it is not a final model.


In [19]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

clean_features = [
    "march_impressions",
    "march_clicks",
    "march_ctr_pct",
    "march_avg_position",
    "march_impression_days",
]
model_df = feature_frame[clean_features + ["future_decline_label"]].copy()

X = model_df[clean_features]
y = model_df["future_decline_label"]

X = pd.DataFrame(
    SimpleImputer(strategy="median").fit_transform(X),
    columns=clean_features,
    index=model_df.index,
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

clean_model = RandomForestClassifier(
    n_estimators=150,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)
clean_model.fit(X_train, y_train)
clean_pred = clean_model.predict_proba(X_test)[:, 1]

clean_auc = roc_auc_score(y_test, clean_pred)
print(f"Clean-feature ROC-AUC: {clean_auc:.3f}")


Clean-feature ROC-AUC: 0.636


In [20]:
# Deliberate leakage: the label itself is exposed as a feature.
leaky_df = model_df.copy()
leaky_df["leaked_decline_score"] = leaky_df["future_decline_label"]

leaky_features = clean_features + ["leaked_decline_score"]
X_leaky = leaky_df[leaky_features]

X_leaky = pd.DataFrame(
    SimpleImputer(strategy="median").fit_transform(X_leaky),
    columns=leaky_features,
    index=leaky_df.index,
)

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leaky, y, test_size=0.20, random_state=42, stratify=y
)

leaky_model = RandomForestClassifier(
    n_estimators=150,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)
leaky_model.fit(Xl_train, yl_train)
leaky_pred = leaky_model.predict_proba(Xl_test)[:, 1]

leaky_auc = roc_auc_score(yl_test, leaky_pred)

print(f"Clean ROC-AUC:  {clean_auc:.3f}")
print(f"Leaky ROC-AUC:  {leaky_auc:.3f}")
print(f"Absolute jump:  {leaky_auc - clean_auc:+.3f}")

assert leaky_auc > clean_auc, "Leakage experiment did not improve the score; inspect the label/feature construction."
assert leaky_auc > 0.98, "Expected the direct label leakage trap to produce a near-perfect score."

print("PASS: exposing the future label creates the expected near-perfect leakage signal.")


Clean ROC-AUC:  0.636
Leaky ROC-AUC:  1.000
Absolute jump:  +0.364
PASS: exposing the future label creates the expected near-perfect leakage signal.


### Leakage verdict

The jump is invalid evidence of model quality. `leaked_decline_score` is derived directly from `future_decline_label`, which is defined using April performance. At the March decision point, April performance is unavailable.

**Action:** remove the leaked feature. The clean feature list above is the only feature set retained for the decision frame.


In [21]:
# Remove the trap. This is the final feature set.
final_feature_frame = feature_frame[
    clean_features + ["client_hash_id", "content_hash_id", "future_decline_label"]
].copy()

assert "leaked_decline_score" not in final_feature_frame.columns
assert len(clean_features) <= 5

print("Final feature columns:")
print(clean_features)
print("Leakage column present in final frame:", "leaked_decline_score" in final_feature_frame.columns)


Final feature columns:
['march_impressions', 'march_clicks', 'march_ctr_pct', 'march_avg_position', 'march_impression_days']
Leakage column present in final frame: False


# 4. Data limits

- The warehouse is an **unbalanced panel**: different clients have different history depth, so a single calendar window is not equally informative for every client.
- GSC availability is not universal; rows without `gsc_data_available IS TRUE` cannot be interpreted as observed search activity.
- The future-decline label is a **defined outcome proxy**, not a business decision or causal claim. It measures an observed change in impressions between two fixed windows.
- A single March→April transition does not establish seasonality or long-run stability. Later validation should use additional time-separated windows.
- The five-feature frame intentionally uses only a small set of observed search signals; it is not a complete representation of content quality or business impact.


In [22]:
# Final self-check: assertions for the contract.
assert grain_check.empty
assert str(count_window.loc[0, "min_report_date"])[:10] == "2026-03-01"
assert str(count_window.loc[0, "max_report_date"])[:10] == "2026-03-31"
assert len(clean_features) == 5
assert "leaked_decline_score" not in final_feature_frame.columns
assert feature_frame["future_decline_label"].isin([0, 1]).all()

print("SELF-CHECK: PASS")
print("• Source grain verified")
print("• March window verified")
print("• Availability checked with IS TRUE")
print("• Five decision-time-safe features retained")
print("• Leakage trap demonstrated and removed")
print("• Limitation stated")
print("• No credentials are stored in notebook source")


SELF-CHECK: PASS
• Source grain verified
• March window verified
• Availability checked with IS TRUE
• Five decision-time-safe features retained
• Leakage trap demonstrated and removed
• Limitation stated
• No credentials are stored in notebook source


# Submission readiness

The required deliverable is this executed notebook:

`work/notebooks/w03_data_contract.ipynb`

Before submission:

1. Run **Runtime → Run all** in Colab.
2. Confirm every cell finishes without error.
3. Confirm the three verification outputs are visible.
4. Confirm the five-feature frame and leakage experiment outputs are visible.
5. Confirm the final frame contains no `leaked_decline_score`.
6. Commit the executed notebook to the repository.
7. Submit the public repository URL on the FlyRank assignment card.

**Important:** the notebook intentionally does not contain an HF token. The token must remain in Colab Secrets as `HF_TOKEN`.
